# Phase 2 — Feature Engineering
**Input:** `Dataset/Processed/df_identity_clean.csv` (output from Phase 1)  
**Output:** `Dataset/Processed/features.csv`  

**What this notebook does:**
1. Encode categorical string columns into integers
2. Build per-user behavioral profiles (action count, error rate, etc.)
3. Normalize numerical features with MinMaxScaler
4. Merge everything into a final feature matrix
5. Sanity check labels and save output

## Cell 1 — Imports & Load Data

In [25]:
import pandas as pd
pd.set_option('display.max_columns', None)

import numpy as np
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
import os

# update paths to match your local setup
INPUT_PATH  = os.path.expanduser("~/Downloads/federated-hgnn-anomaly-detection/Dataset/Processed/df_identity_clean.csv")
OUTPUT_PATH = os.path.expanduser("~/Downloads/federated-hgnn-anomaly-detection/Dataset/Processed/features.csv")

df = pd.read_csv(INPUT_PATH)
print("Loaded shape:", df.shape)
df.head(3)

Loaded shape: (1000, 17)


,userName,sourceIPAddress,eventSource,eventName,type,accessKeyId,sessionContext.attributes.mfaAuthenticated,sessionContext.sessionIssuer.type,awsRegion,eventType,readOnly,errorCode,event_hour,event_dayofweek,is_weekend,is_off_hours,is_anomaly
0,bert-jan,192.168.10.20,iam.amazonaws.com,CreateRole,IAMUser,AKIATFQR7NSC8Q4X20BJ,Unknown,Unknown,us-east-1,AwsApiCall,False,NoError,12,0,0,0,0
1,bert-jan,192.168.10.20,iam.amazonaws.com,GetRole,IAMUser,AKIATFQR7NSC8Q4X20BJ,Unknown,Unknown,us-east-1,AwsApiCall,True,NoError,12,0,0,0,0
2,bert-jan,192.168.10.20,iam.amazonaws.com,ListRolePolicies,IAMUser,AKIATFQR7NSC8Q4X20BJ,Unknown,Unknown,us-east-1,AwsApiCall,True,NoError,12,0,0,0,0


## Cell 2 — Confirm Required Columns Exist

In [26]:
REQUIRED_COLS = [
    'userName', 'sourceIPAddress', 'eventSource', 'eventName',
    'type', 'accessKeyId',
    'sessionContext.attributes.mfaAuthenticated',
    'sessionContext.sessionIssuer.type',
    'awsRegion', 'eventType', 'readOnly', 'errorCode',
    'event_hour', 'event_dayofweek', 'is_weekend', 'is_off_hours',
    'is_anomaly'
]

missing_cols = [c for c in REQUIRED_COLS if c not in df.columns]
if missing_cols:
    print("WARNING — These columns are missing, check Phase 1 output:")
    for c in missing_cols:
        print("  -", c)
else:
    print("All required columns present. Ready to proceed.")

All required columns present. Ready to proceed.


## Cell 3 — Encode Categorical Columns
LabelEncoder maps each unique string to an integer.  
Example: `'benjamin' → 0`, `'bert-jan' → 1`  
We save all encoders in a dict so we can reverse-lookup the original labels later.

In [27]:
CATEGORICAL_COLS = [
    'userName',
    'sourceIPAddress',
    'eventSource',
    'eventName',
    'type',
    'sessionContext.attributes.mfaAuthenticated',
    'sessionContext.sessionIssuer.type',
    'errorCode',
    'eventType',
]

encoders = {}

for col in CATEGORICAL_COLS:
    if col not in df.columns:
        print(f"Skipping '{col}' — not found in dataframe")
        continue
    le = LabelEncoder()
    df[col + '_enc'] = le.fit_transform(df[col].astype(str))
    encoders[col] = le
    print(f"{col}: {len(le.classes_)} unique classes -> encoded")

# readOnly is already boolean — convert to 0/1 directly
if 'readOnly' in df.columns:
    df['readOnly_enc'] = df['readOnly'].astype(int)
    print("readOnly: bool -> int")

print("\nEncoding complete.")

userName: 3 unique classes -> encoded
sourceIPAddress: 14 unique classes -> encoded
eventSource: 22 unique classes -> encoded
eventName: 162 unique classes -> encoded
type: 4 unique classes -> encoded
sessionContext.attributes.mfaAuthenticated: 3 unique classes -> encoded
sessionContext.sessionIssuer.type: 2 unique classes -> encoded
errorCode: 26 unique classes -> encoded
eventType: 2 unique classes -> encoded
readOnly: bool -> int

Encoding complete.


In [28]:
df.head(10)

,userName,sourceIPAddress,eventSource,eventName,type,accessKeyId,sessionContext.attributes.mfaAuthenticated,sessionContext.sessionIssuer.type,awsRegion,eventType,readOnly,errorCode,event_hour,event_dayofweek,is_weekend,is_off_hours,is_anomaly,userName_enc,sourceIPAddress_enc,eventSource_enc,eventName_enc,type_enc,sessionContext.attributes.mfaAuthenticated_enc,sessionContext.sessionIssuer.type_enc,errorCode_enc,eventType_enc,readOnly_enc
0,bert-jan,192.168.10.20,iam.amazonaws.com,CreateRole,IAMUser,AKIATFQR7NSC8Q4X20BJ,Unknown,Unknown,us-east-1,AwsApiCall,False,NoError,12,0,0,0,0,2,3,6,18,2,0,1,14,0,0
1,bert-jan,192.168.10.20,iam.amazonaws.com,GetRole,IAMUser,AKIATFQR7NSC8Q4X20BJ,Unknown,Unknown,us-east-1,AwsApiCall,True,NoError,12,0,0,0,0,2,3,6,116,2,0,1,14,0,1
2,bert-jan,192.168.10.20,iam.amazonaws.com,ListRolePolicies,IAMUser,AKIATFQR7NSC8Q4X20BJ,Unknown,Unknown,us-east-1,AwsApiCall,True,NoError,12,0,0,0,0,2,3,6,135,2,0,1,14,0,1
3,bert-jan,192.168.10.20,iam.amazonaws.com,ListAttachedRolePolicies,IAMUser,AKIATFQR7NSC8Q4X20BJ,Unknown,Unknown,us-east-1,AwsApiCall,True,NoError,12,0,0,0,0,2,3,6,124,2,0,1,14,0,1
4,bert-jan,192.168.10.20,ec2.amazonaws.com,AssociateRouteTable,IAMUser,AKIATFQR7NSC8Q4X20BJ,Unknown,Unknown,us-east-1,AwsApiCall,False,NoError,12,0,0,0,0,2,3,3,2,2,0,1,14,0,0
5,bert-jan,192.168.10.20,ec2.amazonaws.com,DescribeRouteTables,IAMUser,AKIATFQR7NSC8Q4X20BJ,Unknown,Unknown,us-east-1,AwsApiCall,True,NoError,12,0,0,0,0,2,3,3,62,2,0,1,14,0,1
6,bert-jan,192.168.10.20,ec2.amazonaws.com,DescribeRouteTables,IAMUser,AKIATFQR7NSC8Q4X20BJ,Unknown,Unknown,us-east-1,AwsApiCall,True,NoError,12,0,0,0,0,2,3,3,62,2,0,1,14,0,1
7,bert-jan,192.168.10.20,sts.amazonaws.com,AssumeRole,IAMUser,AKIATFQR7NSC8Q4X20BJ,Unknown,Unknown,us-east-1,AwsApiCall,True,NoError,12,0,0,0,1,2,3,21,3,2,0,1,14,0,1
8,bert-jan,192.168.10.20,sts.amazonaws.com,AssumeRole,IAMUser,AKIATFQR7NSC8Q4X20BJ,Unknown,Unknown,us-east-1,AwsApiCall,True,NoError,12,0,0,0,1,2,3,21,3,2,0,1,14,0,1
9,bert-jan,192.168.10.20,ec2.amazonaws.com,DescribeVpcClassicLinkDnsSupport,IAMUser,AKIATFQR7NSC8Q4X20BJ,Unknown,Unknown,us-east-1,AwsApiCall,True,NoError,12,0,0,0,0,2,3,3,72,2,0,1,14,0,1


In [29]:
print(df.shape)

print(df.columns)

(1000, 27)
Index(['userName', 'sourceIPAddress', 'eventSource', 'eventName', 'type',
       'accessKeyId', 'sessionContext.attributes.mfaAuthenticated',
       'sessionContext.sessionIssuer.type', 'awsRegion', 'eventType',
       'readOnly', 'errorCode', 'event_hour', 'event_dayofweek', 'is_weekend',
       'is_off_hours', 'is_anomaly', 'userName_enc', 'sourceIPAddress_enc',
       'eventSource_enc', 'eventName_enc', 'type_enc',
       'sessionContext.attributes.mfaAuthenticated_enc',
       'sessionContext.sessionIssuer.type_enc', 'errorCode_enc',
       'eventType_enc', 'readOnly_enc'],
      dtype='str')


## Cell 4 — Build User Behavior Profile
Instead of looking at each event in isolation, we compute aggregate statistics per user.  
This captures behavioral patterns — e.g. how many services does this user normally access?  
These statistics become the **node feature vectors** for User nodes in the HGNN graph.

In [30]:
user_profile = df.groupby('userName').agg(
    action_count = ('eventName', 'count'),
    unique_ips = ('sourceIPAddress', 'nunique'),
    unique_services = ('eventSource', 'nunique'),
    unique_events = ('eventName', 'nunique'),
    error_count = ('errorCode', lambda x: (x != 'NoError').sum()),
    off_hours_count = ('is_off_hours', 'sum'),
    weekend_count = ('is_weekend', 'sum'),
    readonly_ratio = ('readOnly', lambda x: x.astype(int).mean()),
).reset_index()

# Derived rate features — more meaningful than raw counts for comparing users
user_profile['error_rate'] = (user_profile['error_count'] / user_profile['action_count']).round(4)
user_profile['off_hours_rate'] = (user_profile['off_hours_count'] / user_profile['action_count']).round(4)
user_profile['weekend_rate'] = (user_profile['weekend_count'] / user_profile['action_count']).round(4)

# Anomaly label per user: flag user as anomalous if ANY of their events was anomalous
user_anomaly = df.groupby('userName')['is_anomaly'].max().reset_index()
user_anomaly.columns = ['userName', 'user_is_anomaly']
user_profile = user_profile.merge(user_anomaly, on='userName', how='left')

print("User behavior profiles:")
user_profile

User behavior profiles:


,userName,action_count,unique_ips,unique_services,unique_events,error_count,off_hours_count,weekend_count,readonly_ratio,error_rate,off_hours_rate,weekend_rate,user_is_anomaly
0,Unknown,71,8,4,15,45,0,0,0.887324,0.6338,0.0,0.0,1
1,benjamin,58,4,4,15,8,0,0,1.000000,0.1379,0.0,0.0,0
2,bert-jan,871,5,21,144,65,0,0,0.778416,0.0746,0.0,0.0,1


## Cell 5 — Normalize Numerical Features
MinMaxScaler squeezes all values into the range [0, 1].  
This prevents `action_count` (e.g. 2642) from dominating `error_rate` (e.g. 0.09).

In [31]:
NUMERICAL_COLS = [
    'action_count',
    'unique_ips',
    'unique_services',
    'unique_events',
    'error_rate',
    'off_hours_rate',
    'weekend_rate',
    'readonly_ratio',
]

scaler = MinMaxScaler()
user_profile[NUMERICAL_COLS] = scaler.fit_transform(user_profile[NUMERICAL_COLS])

print("After normalization (all values should be between 0.0 and 1.0):")
user_profile[NUMERICAL_COLS].describe().round(4)

After normalization (all values should be between 0.0 and 1.0):


,action_count,unique_ips,unique_services,unique_events,error_rate,off_hours_rate,weekend_rate,readonly_ratio
count,3.0000,3.0000,3.0000,3.0000,3.0000,3.0,3.0,3.0000
mean,0.3387,0.4167,0.3333,0.3333,0.3711,0.0,0.0,0.4972
std,0.5728,0.5204,0.5774,0.5774,0.5476,0.0,0.0,0.5000
min,0.0000,0.0000,0.0000,0.0000,0.0000,0.0,0.0,0.0000
25%,0.0080,0.1250,0.0000,0.0000,0.0566,0.0,0.0,0.2457
50%,0.0160,0.2500,0.0000,0.0000,0.1132,0.0,0.0,0.4915
75%,0.5080,0.6250,0.5000,0.5000,0.5566,0.0,0.0,0.7457
max,1.0000,1.0000,1.0000,1.0000,1.0000,0.0,0.0,1.0000


## Cell 6 — Merge Profile Back + Build Final Feature Matrix
Each event row now also carries the behavioral profile of the user who triggered it.  
This is the final feature matrix — one row per event, all values as numbers.

In [32]:
df = df.merge(
    user_profile[['userName'] + NUMERICAL_COLS + ['user_is_anomaly']],
    on='userName',
    how='left',
    suffixes=('', '_profile')
)

# Final feature columns for the HGNN
FEATURE_COLS = [
    # Encoded identity
    'userName_enc',
    'sourceIPAddress_enc',
    'eventSource_enc',
    'eventName_enc',
    'type_enc',
    'sessionContext.attributes.mfaAuthenticated_enc',
    'sessionContext.sessionIssuer.type_enc',
    'errorCode_enc',
    'eventType_enc',
    'readOnly_enc',
    # Time features
    'event_hour',
    'event_dayofweek',
    'is_weekend',
    'is_off_hours',
    # User behavior profile (normalized)
    'action_count',
    'unique_ips',
    'unique_services',
    'unique_events',
    'error_rate',
    'off_hours_rate',
    'weekend_rate',
    'readonly_ratio',
    # Label
    'is_anomaly',
]

# Only keep columns that actually exist (safety check)
final_cols = [c for c in FEATURE_COLS if c in df.columns]
df_features = df[final_cols].copy()

print("Final feature matrix shape:", df_features.shape)
print("\nMissing values in final matrix:")
missing = df_features.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "None — all clean.")
print("\nSample:")
df_features.head(5)

Final feature matrix shape: (1000, 23)

Missing values in final matrix:
None — all clean.

Sample:


,userName_enc,sourceIPAddress_enc,eventSource_enc,eventName_enc,type_enc,sessionContext.attributes.mfaAuthenticated_enc,sessionContext.sessionIssuer.type_enc,errorCode_enc,eventType_enc,readOnly_enc,event_hour,event_dayofweek,is_weekend,is_off_hours,action_count,unique_ips,unique_services,unique_events,error_rate,off_hours_rate,weekend_rate,readonly_ratio,is_anomaly
0,2,3,6,18,2,0,1,14,0,0,12,0,0,0,1.0,0.25,1.0,1.0,0.0,0.0,0.0,0.0,0
1,2,3,6,116,2,0,1,14,0,1,12,0,0,0,1.0,0.25,1.0,1.0,0.0,0.0,0.0,0.0,0
2,2,3,6,135,2,0,1,14,0,1,12,0,0,0,1.0,0.25,1.0,1.0,0.0,0.0,0.0,0.0,0
3,2,3,6,124,2,0,1,14,0,1,12,0,0,0,1.0,0.25,1.0,1.0,0.0,0.0,0.0,0.0,0
4,2,3,3,2,2,0,1,14,0,0,12,0,0,0,1.0,0.25,1.0,1.0,0.0,0.0,0.0,0.0,0


In [33]:
df_features.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 23 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   userName_enc                                    1000 non-null   int64  
 1   sourceIPAddress_enc                             1000 non-null   int64  
 2   eventSource_enc                                 1000 non-null   int64  
 3   eventName_enc                                   1000 non-null   int64  
 4   type_enc                                        1000 non-null   int64  
 5   sessionContext.attributes.mfaAuthenticated_enc  1000 non-null   int64  
 6   sessionContext.sessionIssuer.type_enc           1000 non-null   int64  
 7   errorCode_enc                                   1000 non-null   int64  
 8   eventType_enc                                   1000 non-null   int64  
 9   readOnly_enc                                    1000 

## Cell 7 — Sanity Check: Label Distribution
Confirm the anomaly label is non-trivial.  
Healthy range for a prototype: **5–30% anomaly rate**.  
If result is 0%, go back to Phase 1 Cell 6 and check `HIGH_RISK_EVENTS`.

In [34]:
label_counts = df_features['is_anomaly'].value_counts()
total = len(df_features)

print("Label distribution:")
print(f"  Normal   (0): {label_counts.get(0, 0):>5} rows  ({label_counts.get(0,0)/total*100:.1f}%)")
print(f"  Anomaly  (1): {label_counts.get(1, 0):>5} rows  ({label_counts.get(1,0)/total*100:.1f}%)")

print("\nAnomaly events breakdown (what triggered the flag):")
anomaly_events = df[df['is_anomaly'] == 1]['eventName'].value_counts()
print(anomaly_events.to_string())

print("\nAnomaly rate per user:")
print(df.groupby('userName')['is_anomaly'].mean().round(4).to_string())

Label distribution:
  Normal   (0):   936 rows  (93.6%)
  Anomaly  (1):    64 rows  (6.4%)

Anomaly events breakdown (what triggered the flag):
eventName
GetSecretValue        35
AssumeRole            22
CreateLoginProfile     2
AttachRolePolicy       2
CreateAccessKey        2
AttachUserPolicy       1

Anomaly rate per user:
userName
Unknown     0.1549
benjamin    0.0000
bert-jan    0.0608


## Cell 8 — Save Output

In [36]:
# Create output directory if it doesn't exist
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

df_features.to_csv(OUTPUT_PATH, index=False)

print(f"Saved: {OUTPUT_PATH}")
print(f"Shape: {df_features.shape}")
print(f"Columns: {df_features.columns.tolist()}")

Saved: /Users/philberttan/Downloads/federated-hgnn-anomaly-detection/Dataset/Processed/features.csv
Shape: (1000, 23)
Columns: ['userName_enc', 'sourceIPAddress_enc', 'eventSource_enc', 'eventName_enc', 'type_enc', 'sessionContext.attributes.mfaAuthenticated_enc', 'sessionContext.sessionIssuer.type_enc', 'errorCode_enc', 'eventType_enc', 'readOnly_enc', 'event_hour', 'event_dayofweek', 'is_weekend', 'is_off_hours', 'action_count', 'unique_ips', 'unique_services', 'unique_events', 'error_rate', 'off_hours_rate', 'weekend_rate', 'readonly_ratio', 'is_anomaly']
